# Garmin Connect API — Connection Test & Exploration

Interactive testing ground for the `garminconnect` dependency ahead of building the ingestion
pipeline (`src/ingestion/`).

**Goals**

1. Confirm authentication works using credentials from `src/core/config.py` (backed by `.env`).
2. Download **FIT files** for activities, wellness, and workout definitions — confirming all
   three Bronze entity families are FIT-first (per `architecture_guidelines.md` §6).
3. Investigate **activity ↔ scheduled-workout linkage** for Feature 2 (plan adherence matching).
4. Survey the 150+ method API surface so we know what's available for future ingestion work.

**Before running:**

- Copy [`.env.template`](../.env.template) to `.env` at the repo root and fill in
  `GARMIN__EMAIL` / `GARMIN__PASSWORD`.
- Cells make **live calls to your real Garmin Connect account** and write personal data to
  `data/_scratch_garmin_test/` (git-ignored). Nothing here is committed to version control.
- On first login you may be prompted for an MFA code inline (`input()`); afterwards, OAuth
  tokens are cached (`GARMIN__TOKENSTORE_PATH`, default `~/.garminconnect`) so subsequent
  runs skip credential login.

In [3]:
import sys
from importlib.metadata import version as pkg_version
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    raise RuntimeError("Could not locate repository root (pyproject.toml not found).")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

env_path = REPO_ROOT / ".env"
if not env_path.exists():
    raise FileNotFoundError(
        f"No .env file found at {env_path}. Copy .env.template to .env and fill in "
        "your Garmin credentials before running this notebook."
    )

from src.core.config import get_settings

settings = get_settings()

# Scratch area for this notebook's downloads only — never used by the real
# ingestion pipeline (which will write to settings.data.bronze_dir).
SCRATCH_DIR = REPO_ROOT / "data" / "_scratch_garmin_test"
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)

gc_version = pkg_version("garminconnect")
print(f"Repo root:          {REPO_ROOT}")
print(f"garminconnect:      v{gc_version}")
print(f"Garmin account:     {'CONFIGURED' if settings.garmin.email else 'MISSING'}")
print(f"Token cache:        {settings.garmin.tokenstore_path}")
print(f"Scratch dir:        {SCRATCH_DIR}")

Repo root:          c:\Users\gabri\workspace\strider
garminconnect:      v0.3.16
Garmin account:     CONFIGURED
Token cache:        C:\Users\gabri\.garminconnect
Scratch dir:        c:\Users\gabri\workspace\strider\data\_scratch_garmin_test


## Authentication

`Garmin.login()` uses the same mobile SSO flow as the Android app. Since v0.3.0 the library
ships a custom client with `curl_cffi` for TLS fingerprint impersonation (replacing the older
`garth` dependency — token files from `garth` are **not compatible**). The library tries up to
five authentication strategies in sequence: mobile-iOS via `curl_cffi`, mobile-iOS via
`requests`, SSO-embed-widget, portal-web with Cloudflare delays, and plain-HTTP fallback.

Passing a `tokenstore` path lets the library cache DI OAuth tokens on disk and auto-refresh
them proactively, so re-running this notebook later won't require a fresh login.

**⚠️ Garmin's SSO login endpoint is IP-rate-limited.** A fresh credential login (no cached
tokens yet) can trigger a `429 Too Many Requests` on the very first attempt — the library's
multi-strategy chain issues several requests, compounding the risk. Re-running a failed login
immediately makes this worse, not better.

To avoid this, authentication goes through `src/ingestion/garmin_client.py`, which:

- **Skips `login()`** entirely if this session is already authenticated (safe to re-run this
  cell — no network call).
- **Persists a cooldown marker** next to the token cache after any 429, and refuses further
  login attempts (even across kernel restarts) until the cooldown elapses — see
  `rate_limit.login_cooldown_seconds` in `src/core/config.py` (default 1 hour).
- Once tokens are cached successfully, subsequent runs load them from disk with no fresh
  SSO call.

In [4]:
from garminconnect import (
    Garmin,
    GarminConnectAuthenticationError,
    GarminConnectTooManyRequestsError,
)

from src.ingestion.garmin_client import GarminAuthCooldownError, build_client, login

client = build_client(settings)

try:
    login(client, settings)
    print(
        "Authenticated"
        + (" (session already active)." if client.client.is_authenticated else ".")
    )
except GarminAuthCooldownError as e:
    print(f"Login blocked by cooldown from a prior rate-limit: {e}")
    raise
except GarminConnectAuthenticationError as e:
    print(f"Authentication failed — check GARMIN__EMAIL / GARMIN__PASSWORD in .env: {e}")
    raise
except GarminConnectTooManyRequestsError as e:
    print(f"Rate limited by Garmin Connect — cooldown recorded, wait before retrying: {e}")
    raise

Authenticated (session already active).


## Profile & daily stats (JSON)

Quick sanity checks that the session works, using cheap, low-volume endpoints.

In [5]:
from datetime import date, timedelta

yesterday = (date.today() - timedelta(days=1)).isoformat()

print("Unit system:", client.get_unit_system())

stats = client.get_stats(yesterday)
print(f"\nYesterday ({yesterday}):")
print(f"  Steps:        {stats.get('totalSteps')}")
print(f"  Calories:     {stats.get('totalKilocalories')}")
print(f"  Resting HR:   {stats.get('restingHeartRate')}")
print(f"  Stress avg:   {stats.get('averageStressLevel')}")

Unit system: metric

Yesterday (2026-09-26):
  Steps:        11511
  Calories:     2336.0
  Resting HR:   56
  Stress avg:   26


## Activities & FIT downloads (primary focus)

`download_activity(activity_id, dl_fmt)` returns raw bytes in the requested format.

**Important:** the default format is `TCX`, not `ORIGINAL`. To get the original
device-recorded FIT file (zipped), you must explicitly pass
`Garmin.ActivityDownloadFormat.ORIGINAL`. Other formats (`TCX`, `GPX`, `KML`, `CSV`) are
lossy derivatives and not used by the pipeline.

Per `architecture_guidelines.md`, FIT is the Bronze-layer source of truth for activities;
`parser_fit.py` (via `fitdecode`) will consume the binary FIT directly for Silver layer
transformation.

In [6]:
total = client.count_activities()
print(f"Total activities on account: {total}\n")

recent = client.get_activities(start=0, limit=5)
for act in recent:
    activity_type = act.get("activityType", {}).get("typeKey", "?")
    print(
        f"{act['activityId']:>12}  {act.get('startTimeLocal', '?'):>20}  "
        f"{act.get('activityName', '?'):<30}  {activity_type}"
    )

Total activities on account: 1248

 24520141087   2026-09-27 15:28:08  Lodi Walking                    walking
 24513761653   2026-09-27 09:39:41  Lodi Walking                    walking
 24507581501   2026-09-26 17:17:46  Lodi Walking                    walking
 24501751461   2026-09-26 10:06:06  Lodi Walking                    walking
 24501749482   2026-09-26 00:20:07  Lodi Walking                    walking


In [7]:
import zipfile

if not recent:
    print("No activities on the account to download.")
else:
    activity_id = recent[0]["activityId"]
    activity_name = recent[0].get("activityName", "unnamed")

    fit_zip_bytes = client.download_activity(
        activity_id, dl_fmt=Garmin.ActivityDownloadFormat.ORIGINAL
    )
    zip_path = SCRATCH_DIR / f"activity_{activity_id}.zip"
    zip_path.write_bytes(fit_zip_bytes)
    print(f"'{activity_name}' -> {len(fit_zip_bytes):,} bytes saved to {zip_path}")

    with zipfile.ZipFile(zip_path) as zf:
        for name in zf.namelist():
            size = zf.getinfo(name).file_size
            print(f"  contains: {name} ({size:,} bytes)")

'Lodi Walking' -> 59,873 bytes saved to c:\Users\gabri\workspace\strider\data\_scratch_garmin_test\activity_24520141087.zip
  contains: 24520141087_ACTIVITY.fit (158,767 bytes)


## Wellness / monitoring data

Day-level wellness metrics are available as **JSON** via dedicated endpoints (`get_heart_rates`,
`get_sleep_data`, `get_stress_data`, etc.).

However, `download_health_snapshot(date)` calls `/download-service/files/wellness/{date}` and
returns a **ZIP of binary FIT files** for the day — including `WELLNESS` (steps/HR/stress
epochs), `SLEEP_DATA`, and `METRICS`.

**Architecture decision (confirmed):** wellness Bronze storage uses binary FIT from the health
snapshot endpoint, not the JSON endpoints. JSON endpoints remain useful for quick ad-hoc reads
in notebooks but are not the Bronze source of truth. If a metric is JSON-only (no FIT
equivalent), save the JSON in Bronze.

In [8]:
wellness_calls = {
    "get_heart_rates": client.get_heart_rates,
    "get_sleep_data": client.get_sleep_data,
    "get_stress_data": client.get_stress_data,
    "get_hrv_data": client.get_hrv_data,
    "get_body_battery": client.get_body_battery,
    "get_spo2_data": client.get_spo2_data,
    "get_respiration_data": client.get_respiration_data,
    "get_training_status": client.get_training_status,
    "get_training_readiness": client.get_training_readiness,
}

for name, fn in wellness_calls.items():
    try:
        result = fn(yesterday)
        if isinstance(result, dict):
            shape = f"dict, keys: {list(result.keys())}"
        elif isinstance(result, list):
            shape = f"list[{len(result)}]"
        elif result is None:
            shape = "None"
        else:
            shape = type(result).__name__
        print(f"{name:<24} -> {shape}")
    except Exception as e:
        print(f"{name:<24} -> ERROR: {e}")

get_heart_rates          -> dict, keys: ['userProfilePK', 'calendarDate', 'startTimestampGMT', 'endTimestampGMT', 'startTimestampLocal', 'endTimestampLocal', 'maxHeartRate', 'minHeartRate', 'restingHeartRate', 'lastSevenDaysAvgRestingHeartRate', 'heartRateValues', 'heartRateValueDescriptors']
get_sleep_data           -> dict, keys: ['dailySleepDTO', 'sleepMovement', 'remSleepData', 'sleepLevels', 'wellnessSpO2SleepSummaryDTO', 'sleepStress', 'sleepBodyBattery', 'skinTempDataExists', 'restingHeartRate']
get_stress_data          -> dict, keys: ['userProfilePK', 'calendarDate', 'startTimestampGMT', 'endTimestampGMT', 'startTimestampLocal', 'endTimestampLocal', 'maxStressLevel', 'avgStressLevel', 'stressChartValueOffset', 'stressChartYAxisOrigin', 'stressValueDescriptorsDTOList', 'stressValuesArray', 'bodyBatteryValueDescriptorsDTOList', 'bodyBatteryValuesArray']
get_hrv_data             -> dict, keys: []
get_body_battery         -> list[1]
get_spo2_data            -> dict, keys: ['userPro

In [9]:
from garminconnect import GarminConnectConnectionError, GarminConnectNotFoundError

# Today's snapshot may not be finalized yet server-side; try yesterday then day before.
candidate_dates = [yesterday, (date.today() - timedelta(days=2)).isoformat()]

snapshot_bytes = None
snapshot_date = None
for d in candidate_dates:
    try:
        snapshot_bytes = client.download_health_snapshot(d)
        snapshot_date = d
        break
    except (GarminConnectNotFoundError, GarminConnectConnectionError) as e:
        print(f"No snapshot for {d}: {e}")

if snapshot_bytes is None:
    print("No health/wellness snapshot available for the recent dates tried.")
else:
    snapshot_path = SCRATCH_DIR / f"wellness_{snapshot_date}.zip"
    snapshot_path.write_bytes(snapshot_bytes)
    print(f"Saved {len(snapshot_bytes):,} bytes -> {snapshot_path}")

    with zipfile.ZipFile(snapshot_path) as zf:
        fit_count = sum(1 for n in zf.namelist() if n.lower().endswith(".fit"))
        print(f"Contains {len(zf.namelist())} files ({fit_count} binary .fit):\n")
        for name in zf.namelist():
            size = zf.getinfo(name).file_size
            print(f"  {name} ({size:,} bytes)")

Saved 48,293 bytes -> c:\Users\gabri\workspace\strider\data\_scratch_garmin_test\wellness_2026-09-26.zip
Contains 24 files (24 binary .fit):

  486683336835_WELLNESS.fit (2,691 bytes)
  486829083475_WELLNESS.fit (774 bytes)
  486828560362_WELLNESS.fit (3,370 bytes)
  486808992804_WELLNESS.fit (10,572 bytes)
  486683329263_WELLNESS.fit (582 bytes)
  486713799966_SLEEP_DATA.fit (1,948 bytes)
  486809008938_SLEEP_DATA.fit (5,314 bytes)
  486683385455_SLEEP_DATA.fit (178 bytes)
  486683399247_SLEEP_DATA.fit (742 bytes)
  486683382252_SLEEP_DATA.fit (4,324 bytes)
  486683389194_SLEEP_DATA.fit (178 bytes)
  486828563299_SLEEP_DATA.fit (1,543 bytes)
  486808994927_METRICS.fit (491 bytes)
  486943066864_WELLNESS.fit (10,483 bytes)
  486943149651_SLEEP_DATA.fit (5,830 bytes)
  486683331730_WELLNESS.fit (558 bytes)
  486683351472_METRICS.fit (491 bytes)
  486713795885_WELLNESS.fit (4,478 bytes)
  486829087340_SLEEP_DATA.fit (229 bytes)
  486683342108_METRICS.fit (452 bytes)
  486683359484_METRIC

## Workouts, plans & activity ↔ workout linkage

The Garmin Connect API exposes the **full workout/plan ecosystem**:

- **Scheduled calendar:** `get_scheduled_workouts(year, month)` returns a calendar dict with
  `calendarItems` — each workout-type item has `workoutId`, `atpPlanId`, `title`, `date`,
  and `sportTypeKey`. This is JSON-only (no FIT equivalent for the schedule metadata).
- **Workout definitions:** `get_workout_by_id(id)` returns structured JSON
  (`workoutSegments[].workoutSteps[]` with step types, end conditions, and targets).
  `download_workout(id)` returns the raw **binary FIT** — not zipped, unlike activities.
- **Activity ↔ workout linkage:** activities started from a scheduled workout carry an
  explicit `workoutId` field in the activity list payload, plus
  `metadataDTO.associatedWorkoutId` in the per-activity detail. This **exactly matches**
  the calendar item's `workoutId`, making plan adherence matching a direct join rather than
  a fuzzy heuristic. Freeform activities have `workoutId: None`.

This section validates all three claims.

In [10]:
import json

today_d = date.today()

# Calendar view for the current month
workout_items = []
try:
    scheduled_month = client.get_scheduled_workouts(today_d.year, today_d.month)
    workout_items = [
        i for i in scheduled_month.get("calendarItems", [])
        if i.get("itemType") == "workout"
    ]
    print(f"Workout items this month: {len(workout_items)}\n")
    for item in workout_items:
        print(
            f"  {item['date']}  {item.get('title', '?'):<20}  "
            f"workoutId={item.get('workoutId')}"
        )
except Exception as e:
    print(f"get_scheduled_workouts -> ERROR: {e}")

# Next scheduled workout
next_workout = None
try:
    next_workout = client.get_next_scheduled_workout()
    print(
        f"\nNext scheduled: '{next_workout.get('title')}' on {next_workout.get('date')}"
        f" (workoutId={next_workout.get('workoutId')})"
    )
except Exception as e:
    print(f"get_next_scheduled_workout -> ERROR: {e}")

Workout items this month: 14

  2026-09-07  Benchmark Run         workoutId=1690884303
  2026-09-09  Easy Run              workoutId=1691034470
  2026-09-11  Easy Run              workoutId=1691034471
  2026-09-13  Easy Run              workoutId=1691034475
  2026-09-14  Easy Run              workoutId=1691034479
  2026-09-16  Easy Run              workoutId=1692721049
  2026-09-18  Stride Repeats        workoutId=1694263647
  2026-09-20  Long Easy Run         workoutId=1697686883
  2026-09-21  Easy Run              workoutId=1698542601
  2026-09-24  Speed Repeats         workoutId=1700418354
  2026-09-27  Easy Run              workoutId=1701701439
  2026-09-28  Easy Run              workoutId=1705470783
  2026-09-29  Progression Run       workoutId=1708271876
  2026-10-01  Stride Repeats        workoutId=1710498196

Next scheduled: 'Easy Run' on 2026-09-27 (workoutId=1701701439)


In [11]:
# Cross-reference past scheduled workouts against completed activities.
past_items = [i for i in workout_items if i["date"] < today_d.isoformat()]
acts_by_date = {}

if past_items:
    month_activities = client.get_activities_by_date(
        past_items[0]["date"], past_items[-1]["date"], activitytype="running"
    )
    acts_by_date = {a["startTimeLocal"][:10]: a for a in month_activities}

    print(f"{'planned date':<12} {'title':<20} {'matched?':<14} activity name / duration")
    print("-" * 85)
    for item in past_items:
        match = acts_by_date.get(item["date"])
        if match:
            mins = round(match["duration"] / 60, 1)
            wid = match.get("workoutId")
            tag = "YES (ID)" if wid and wid == item.get("workoutId") else "YES (date)"
            print(
                f"{item['date']:<12} {item['title']:<20} {tag:<14} "
                f"'{match['activityName']}' ({mins} min)"
            )
        else:
            print(f"{item['date']:<12} {item['title']:<20} NO MATCH")
else:
    print("No past workout items found this month.")

planned date title                matched?       activity name / duration
-------------------------------------------------------------------------------------
2026-09-07   Benchmark Run        YES (ID)       'Lodi - Benchmark Run' (9.0 min)
2026-09-09   Easy Run             YES (ID)       'Lodi - Easy Run' (30.0 min)
2026-09-11   Easy Run             YES (ID)       'Lodi - Easy Run' (30.0 min)
2026-09-13   Easy Run             NO MATCH
2026-09-14   Easy Run             YES (ID)       'Lodi - Easy Run' (30.0 min)
2026-09-16   Easy Run             YES (ID)       'Lodi - Easy Run' (30.1 min)
2026-09-18   Stride Repeats       YES (ID)       'Lodi - Stride Repeats' (36.5 min)
2026-09-20   Long Easy Run        YES (ID)       'Varazze - Long Easy Run' (70.0 min)
2026-09-21   Easy Run             YES (ID)       'Lodi - Easy Run' (38.5 min)
2026-09-24   Speed Repeats        YES (ID)       'Lodi - Speed Repeats' (36.8 min)


In [12]:
def find_keys_containing(obj, needle, path=""):
    """Recursively yield (path, value) for keys containing `needle` (case-insensitive)."""
    hits = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            new_path = f"{path}.{k}" if path else k
            if needle.lower() in k.lower():
                hits.append((new_path, v))
            hits.extend(find_keys_containing(v, needle, new_path))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            hits.extend(find_keys_containing(v, needle, f"{path}[{i}]"))
    return hits


# Pick a workout-linked activity and verify the workoutId linkage end-to-end.
linked_date = next(
    (i["date"] for i in past_items if i["date"] in acts_by_date), None
)

if linked_date:
    linked_activity = acts_by_date[linked_date]
    linked_item = next(i for i in past_items if i["date"] == linked_date)

    print(f"Inspecting '{linked_activity.get('activityName')}'"
          f" (id={linked_activity['activityId']})\n")

    # List-level payload already carries workoutId
    print(f"List-level workoutId:         {linked_activity.get('workoutId')}")
    print(f"Scheduled calendar workoutId: {linked_item.get('workoutId')}")

    # Per-activity detail carries it under metadataDTO
    detail = client.get_activity(linked_activity["activityId"])
    assoc_wid = detail.get("metadataDTO", {}).get("associatedWorkoutId")
    print(f"metadataDTO.associatedWorkoutId: {assoc_wid}")
    print(
        f"All three match: "
        f"{linked_activity.get('workoutId') == linked_item.get('workoutId') == assoc_wid}"
    )

    # Confirm freeform activities have workoutId=None
    freeform = next((a for a in recent if not a.get("workoutId")), None)
    if freeform:
        print(
            f"\nFreeform activity ('{freeform.get('activityName')}'): "
            f"workoutId={freeform.get('workoutId')}"
        )
else:
    print("No workout-linked activity found for cross-validation.")

Inspecting 'Lodi - Benchmark Run' (id=24273474284)

List-level workoutId:         1690884303
Scheduled calendar workoutId: 1690884303
metadataDTO.associatedWorkoutId: 1690884303
All three match: True

Freeform activity ('Lodi Walking'): workoutId=None


In [13]:
# Workout definition: JSON structure + binary FIT download.
sample_workout_id = None
if next_workout:
    sample_workout_id = next_workout.get("workoutId")
elif past_items:
    sample_workout_id = past_items[-1].get("workoutId")

if sample_workout_id:
    # Structured JSON definition
    workout_detail = client.get_workout_by_id(sample_workout_id)
    print(f"Workout '{workout_detail.get('workoutName')}' (id={sample_workout_id}):\n")
    print(json.dumps(workout_detail, indent=2, default=str)[:1500])

    # Binary FIT download (raw bytes, NOT zipped unlike activities/wellness)
    workout_bytes = client.download_workout(sample_workout_id)
    has_fit_header = b".FIT" in workout_bytes[:14]
    is_zip = workout_bytes[:2] == b"PK"
    print(f"\n\ndownload_workout -> {len(workout_bytes):,} bytes")
    print(f"  Binary FIT (raw, not zipped): {has_fit_header and not is_zip}")
else:
    print("No workout ID available to inspect.")

Workout 'Easy Run' (id=1701701439):

{
  "workoutId": 1701701439,
  "ownerId": 89118819,
  "workoutName": "Easy Run",
  "description": "Run at an easy pace.",
  "updatedDate": "2026-09-18T11:33:21.0",
  "createdDate": "2026-09-18T11:33:21.0",
  "sportType": {
    "sportTypeId": 1,
    "sportTypeKey": "running",
    "displayOrder": 1
  },
  "subSportType": null,
  "trainingPlanId": null,
  "author": {
    "userProfilePk": 89118819,
    "displayName": "2773b3cd-4bf5-4e51-bca4-b7ac1e7d1046",
    "fullName": "Gabriele Di Corato",
    "profileImgNameLarge": null,
    "profileImgNameMedium": "4053a852-5eb2-43e5-8af4-2b090314fad1-89118819.png",
    "profileImgNameSmall": "5c0bc975-43a5-4a57-a18d-0dedeb476561-89118819.png",
    "userPro": false,
    "vivokidUser": false
  },
  "sharedWithUsers": null,
  "estimatedDurationInSecs": 2700,
  "estimatedDistanceInMeters": null,
  "workoutSegments": [
    {
      "segmentOrder": 1,
      "sportType": {
        "sportTypeId": 1,
        "sportTypeKey"

## Training & fitness signals

Survey of the training/fitness metrics bucket — race predictions, endurance/hill scores,
lactate threshold, running tolerance, nutrition. These are candidate inputs to Feature 1's
AI action catalog (e.g. "Race Readiness" narration).

**Note on method signatures** (v0.3.16):
- `get_lactate_threshold()` uses **keyword-only** args: `latest=True` by default (no
  positional date).
- `get_running_tolerance(startdate, enddate)` requires **both** start and end dates.
- `get_endurance_score(startdate, enddate=None)` and `get_hill_score(startdate, enddate=None)`
  take a start date and optional end date.
- Several endpoints return empty/null for accounts with limited training history or older
  devices.

In [14]:
_start = (today_d - timedelta(days=30)).isoformat()
_end = today_d.isoformat()

signal_calls = {
    "get_race_predictions": lambda: client.get_race_predictions(),
    "get_training_readiness": lambda: client.get_training_readiness(_end),
    "get_morning_training_readiness": lambda: client.get_morning_training_readiness(_end),
    "get_endurance_score": lambda: client.get_endurance_score(_start, _end),
    "get_hill_score": lambda: client.get_hill_score(_start, _end),
    "get_lactate_threshold": lambda: client.get_lactate_threshold(),
    "get_running_tolerance": lambda: client.get_running_tolerance(_start, _end),
    "get_max_metrics": lambda: client.get_max_metrics(_end),
    "get_nutrition_daily_food_log": lambda: client.get_nutrition_daily_food_log(_end),
    "get_nutrition_daily_meals": lambda: client.get_nutrition_daily_meals(_end),
    "get_nutrition_daily_settings": lambda: client.get_nutrition_daily_settings(_end),
}

for name, fn in signal_calls.items():
    try:
        result = fn()
        if isinstance(result, dict):
            shape = f"dict, keys: {list(result.keys())[:10]}"
        elif isinstance(result, list):
            shape = f"list[{len(result)}]"
        elif result is None:
            shape = "None"
        else:
            shape = f"{type(result).__name__}: {result}"
        print(f"{name:<34} -> {shape}")
    except Exception as e:
        print(f"{name:<34} -> ERROR: {e}")

get_race_predictions               -> dict, keys: ['userId', 'fromCalendarDate', 'toCalendarDate', 'calendarDate', 'time5K', 'time10K', 'timeHalfMarathon', 'timeMarathon']
get_training_readiness             -> list[0]
get_morning_training_readiness     -> None
get_endurance_score                -> dict, keys: ['userProfilePK', 'startDate', 'endDate', 'avg', 'max', 'groupMap', 'enduranceScoreDTO']
get_hill_score                     -> dict, keys: ['userProfilePK', 'startDate', 'endDate', 'periodAvgScore', 'maxScore', 'hillScoreDTOList']
get_lactate_threshold              -> dict, keys: ['speed_and_heart_rate', 'power']
get_running_tolerance              -> list[0]
get_max_metrics                    -> list[0]
get_nutrition_daily_food_log       -> dict, keys: ['mealDate', 'dayStartTime', 'dayEndTime', 'dailyViewType', 'mealDetails', 'loggedFoodsWithServingSizes']
get_nutrition_daily_meals          -> dict, keys: ['meals', 'dailyViewType']
get_nutrition_daily_settings       -> dict, keys:

## Broader API surface

`garminconnect` exposes 150+ methods on the `Garmin` class. Rather than hardcoding a list
that drifts as the dependency is upgraded, introspect the installed version directly and
bucket the public methods by theme.

Notable capabilities beyond what's exercised above:

- **`garminconnect.workout`** module (requires `garminconnect[workout]`): Pydantic builders
  for structured workouts — `RunningWorkout`, `create_warmup_step`, `create_interval_step`,
  `create_recovery_step`, `create_cooldown_step`, etc. Feature 3's plan-push side will use
  these via `upload_running_workout()` rather than hand-building JSON.
- **`garmin.typed.*`** (requires `garminconnect[typed]`): experimental Pydantic response
  models for selected endpoints (`DailyStats`, `SleepData`, `Activity`, etc.).
- **`parse_activity_detail_metrics(details)`**: helper to map positional arrays in
  `activityDetailMetrics` into named metric dicts (`directHeartRate`, `directSpeed`, etc.).
- **`connectapi(path, **kwargs)`** and **`query_garmin_graphql(query)`**: low-level escape
  hatches for endpoints not yet wrapped by named methods.

In [15]:
import inspect

public_methods = sorted(
    name
    for name, member in inspect.getmembers(Garmin, predicate=inspect.isfunction)
    if not name.startswith("_")
)
print(f"Total public methods on Garmin (v{gc_version}): {len(public_methods)}")

keyword_categories = {
    "activities": ("activit",),
    "wellness/health": (
        "sleep", "stress", "hrv", "spo2", "respiration", "battery",
        "heart_rate", "heartrate", "hydration", "blood_pressure", "rhr",
    ),
    "training/fitness metrics": (
        "training", "endurance", "race", "fitnessage", "hill_score",
        "max_metrics", "lactate", "ftp", "cycling_ftp", "zones",
        "running_tolerance", "functional_threshold",
    ),
    "body composition/weight": ("weigh", "body_composition"),
    "gear/device": ("gear", "device"),
    "workouts/training plans": ("workout", "training_plan"),
    "goals/badges/challenges": ("goal", "badge", "challenge", "record"),
    "golf": ("golf",),
    "menstrual/pregnancy": ("menstrual", "pregnancy"),
    "nutrition": ("nutrition",),
    "account/system": (
        "login", "logout", "profile", "graphql", "download",
        "connectapi", "typed", "reload", "resume",
    ),
}

categorized: set[str] = set()
categories: dict[str, list[str]] = {}
for label, keywords in keyword_categories.items():
    matches = [m for m in public_methods if any(k in m for k in keywords)]
    categories[label] = matches
    categorized.update(matches)
categories["other"] = [m for m in public_methods if m not in categorized]

for label, methods in categories.items():
    print(f"\n{label} ({len(methods)})")
    for m in methods:
        print(f"  - {m}")

Total public methods on Garmin (v0.3.16): 159

activities (29)
  - add_gear_to_activity
  - count_activities
  - create_manual_activity
  - create_manual_activity_from_json
  - delete_activity
  - download_activity
  - get_activities
  - get_activities_by_date
  - get_activities_fordate
  - get_activity
  - get_activity_details
  - get_activity_exercise_sets
  - get_activity_gear
  - get_activity_hr_in_timezones
  - get_activity_power_in_timezones
  - get_activity_split_summaries
  - get_activity_splits
  - get_activity_typed_splits
  - get_activity_types
  - get_activity_weather
  - get_gear_activities
  - get_last_activity
  - import_activity
  - remove_gear_from_activity
  - set_activity_description
  - set_activity_exercise_sets
  - set_activity_name
  - set_activity_type
  - upload_activity

wellness/health (20)
  - add_hydration_data
  - delete_blood_pressure
  - get_all_day_stress
  - get_blood_pressure
  - get_body_battery
  - get_body_battery_events
  - get_heart_rate_zones
  

## Findings & architecture implications

### All three Bronze entity families are FIT-first

| Entity | Method | Format | Notes |
|--------|--------|--------|-------|
| Activities | `download_activity(id, dl_fmt=ORIGINAL)` | FIT, zipped | Default is `TCX` — must pass `ORIGINAL` explicitly |
| Wellness/monitoring | `download_health_snapshot(date)` | Multiple FIT per day, zipped | Contains `WELLNESS`, `SLEEP_DATA`, `METRICS` files |
| Workout definitions | `download_workout(id)` | Raw FIT, **not** zipped | Smallest/cleanest of the three downloads |

### Activity ↔ workout linkage is a direct join

- Activities started from a scheduled workout carry `workoutId` in the **list** payload
  (cheap batch call) and `metadataDTO.associatedWorkoutId` in the per-activity detail.
  Both exactly match the scheduled calendar item's `workoutId`.
- Freeform activities have `workoutId: None` and `isAtpActivity: False` — a clean
  "not planned" signal.
- **Feature 2 matching strategy:** join on `workoutId`. Fall back to date-window + type +
  name-substring matching only for genuinely freeform activities. The LLM-assisted
  ambiguous-case path is a last resort, not the common case.

### Signal availability varies by account/device

- `get_race_predictions()` returns real 5K/10K/HM/Marathon time estimates — good MVP action
  candidate ("Race Readiness" narration).
- `get_training_readiness`, `get_endurance_score`, `get_hill_score` may return empty on
  accounts with limited training history or older devices.
- `get_lactate_threshold()` uses **keyword-only** args (`latest=True` default) — no
  positional date param.
- `get_running_tolerance(startdate, enddate)` requires **both** start and end dates.
- Nutrition endpoints are gated on the user maintaining a Garmin Connect food log.

### Feature 3 can use `garminconnect.workout` Pydantic builders

The `upload_running_workout()` method accepts structured `RunningWorkout` objects built from
`garminconnect.workout` helpers (`create_warmup_step`, `create_interval_step`, etc.) — this
is the push path for AI-generated plans, rather than hand-crafting workout JSON.

### Next steps

Build `src/ingestion/garmin_client.py` data-fetching methods (thin adapter wrapping the calls
exercised above) and `src/ingestion/ledger.py` (hash/timestamp tracking per `activity_id` /
date so re-syncs are safe and idempotent).